In [1]:
# =====================================
# IMPORTACIÓN DE LIBRERÍAS
# =====================================

import sys
import os
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt 
import streamlit as st 

# =====================================
# CARGA DE DATOS
# =====================================

# sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "../.."))) # ESTA LINEA PARA PY EN JUPYTER NO FUNCIONA
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../.."))) # ESTA LINEA SOLO APRA JUPITER EN PY USAR LA DE ARRIBA
from src.utils.constantes import DATA_CLEAN_PATH



In [ ]:

# Abrir el Archivo

archivo_clean_path = DATA_CLEAN_PATH / "usu_clean_individual.csv"
df = pd.read_csv(archivo_clean_path, delimiter=",", low_memory=False)

# =====================================
# ENCABEZADO DE LA APLICACIÓN
# =====================================

#st.title("1.6 (P6) EDUCACIÓN")
#st.subheader("Visualizacion de informacion de educacion a nivel población segun la Encuesta Permanente de Hogares (EPH)")

# =====================================
# SELECCIÓN DE AÑO Y TRIMESTRE
# =====================================

def seleccionar_anio_trimestre(df):
    """
    Muestra selectores de Streamlit para que el usuario elija un año y trimestre disponibles en el DataFrame.
    Devuelve el DataFrame filtrado según esa selección junto con los valores seleccionados de año y trimestre.
    """
    anios = sorted(df["ANO4"].astype(str).unique())
    trimestres = sorted(df['TRIMESTRE'].astype(str).unique())

    anio = st.selectbox("Seleccione el año", options=anios)
    if str(anio) not in anios:
        st.error("No se encontró el año en el sistema, seleccione otro")

    trimestre = st.selectbox("Seleccione el trimestre", options=trimestres)

    # Filtrar el DataFrame por año y trimestre seleccionados
    df_filtrado = df[(df["ANO4"] == int(anio)) & (df["TRIMESTRE"] == int(trimestre))]
    return df_filtrado, anio, trimestre

df_filtrado, anio, trimestre = seleccionar_anio_trimestre(df)

In [2]:

# Abrir el Archivo

archivo_clean_path = DATA_CLEAN_PATH / "usu_clean_individual.csv"
df = pd.read_csv(archivo_clean_path, delimiter=",", low_memory=False)

In [6]:
# Filtrar la base para el año 2024 y primer trimestre
df_filtrado = df[(df['ANO4'] == 2024) & (df['TRIMESTRE'] == 1)].copy()

In [4]:
df_filtrado.head(3)

,CODUSU,ANO4,TRIMESTRE,NRO_HOGAR,COMPONENTE,H15,REGION,MAS_500,AGLOMERADO,PONDERA,...,IDECCFR,RDECCFR,GDECCFR,PDECCFR,ADECCFR,PONDIH,CH04_str,NIVEL_ED_str,CONDICION_LABORAL,UNIVERSITARIO
0,TQRMNOPUTHLMKQCDEGGFB00852588,2024,1,1,3,1,42,S,10,439,...,12.0,12,12.0,NaN,12,0,femenino,secundario incompleto,ocupado dependiente,0
1,TQRMNOPUTHLMKQCDEGGFB00852588,2024,1,1,4,1,42,S,10,439,...,12.0,12,12.0,NaN,12,0,femenino,superior o universitario,inactivo,1
2,TQRMNOPUTHLMKQCDEGGFB00852588,2024,1,1,5,0,42,S,10,439,...,12.0,12,12.0,NaN,12,0,femenino,sin informacion,fuera de categoria,0


In [11]:
print(df.dtypes)


CODUSU               object
ANO4                  int64
TRIMESTRE             int64
NRO_HOGAR             int64
COMPONENTE            int64
                      ...  
PONDIH                int64
CH04_str             object
NIVEL_ED_str         object
CONDICION_LABORAL    object
UNIVERSITARIO         int64
Length: 181, dtype: object


In [ ]:
# Ver todos los años y aglomerados disponibles
anios = sorted(df['ANO4'].astype(str).unique())
aglomerados = sorted(df['AGLOMERADO'].astype(str).unique())

In [ ]:
print(anios);
print(aglomerados);

['2023', '2024']
['10', '12', '13', '14', '15', '17', '18', '19', '2', '20', '22', '23', '25', '26', '27', '29', '3', '30', '31', '32', '33', '34', '36', '38', '4', '5', '6', '7', '8', '9', '91', '93']


# 1.5.1 Para las personas desocupadas, informar la cantidad de ellas según sus estudios alcanzados.
#  Se debe informar para un año y trimestre elegido por el usuario.
Columnas a usar:
PONDERA
NIVEL_ED N (1) Nivel educativo
1 = Primario incompleto (incluye
educación especial)
2 = Primario completo
3 = Secundario incompleto
4 = Secundario completo
5 = Superior universitario incompleto
6 = Superior universitario completo
7 = Sin instrucción
9 = Ns/Nr

ESTADO N (1) Condición de actividad
0 = Entrevista individual no realizada
(no respuesta al cuestionariob individual)
1 = Ocupado
2 = Desocupado
3 = Inactivo
4 = Menor de 10 años

In [7]:
# Mapas de significado
estado_map = {
    0: 'No respondió',
    1: 'Ocupado',
    2: 'Desocupado',
    3: 'Inactivo',
    4: 'Menor de 10 años'
}

nivel_ed_map = {
    1: 'Primario incompleto',
    2: 'Primario completo',
    3: 'Secundario incompleto',
    4: 'Secundario completo',
    5: 'Superior univ. incompleto',
    6: 'Superior univ. completo',
    7: 'Sin instrucción',
    9: 'Ns/Nr'
}

# Aplicar los mapas
df_filtrado['ESTADO_DESC'] = df_filtrado['ESTADO'].map(estado_map)
df_filtrado['NIVEL_ED_DESC'] = df_filtrado['NIVEL_ED'].map(nivel_ed_map)

# Filtrar por personas mayores de 10 años (excluimos ESTADO = 0 y 4)
df_adultos = df_filtrado[df_filtrado['ESTADO'].isin([1, 2, 3])]

# Tabla cruzada ponderada: Nivel educativo vs Condición de actividad
tabla = df_adultos.pivot_table(
    index='NIVEL_ED_DESC',
    columns='ESTADO_DESC',
    values='PONDERA',
    aggfunc='sum',
    fill_value=0
)

# Ordenar por total (opcional)
tabla['Total'] = tabla.sum(axis=1)
tabla = tabla.sort_values(by='Total', ascending=False)
tabla = tabla.drop(columns='Total')  # quitamos columna auxiliar si no la querés ver

# Mostrar tabla
tabla

ESTADO_DESC,Desocupado,Inactivo,Ocupado
NIVEL_ED_DESC,,,
Secundario completo,413970,1747055,4129791
Secundario incompleto,227082,3753918,2097867
Superior univ. completo,100192,876213,3269866
Superior univ. incompleto,197134,1404510,1932772
Primario completo,125909,1515505,1335554
Primario incompleto,23955,1778717,317091
Sin instrucción,0,120783,30987


1.5.2 Informar la evolución del desempleo(tasa de desempleo) a lo largo del tiempo. Se debe poder fi ltrar por aglomerado y en caso de no elegir ninguno se debe calcular para todo el país.
La tasa de desempleo es el cociente de personas desocupadas y la suma de personas desocupadas más ocupadas multiplicado por 100.

Columnas a usar:
PONDERA
NIVEL_ED N (1) Nivel educativo
1 = Primario incompleto (incluye
educación especial)
2 = Primario completo
3 = Secundario incompleto
4 = Secundario completo
5 = Superior universitario incompleto
6 = Superior universitario completo
7 = Sin instrucción
9 = Ns/Nr

ESTADO N (1) Condición de actividad
0 = Entrevista individual no realizada
(no respuesta al cuestionariob individual)
1 = Ocupado
2 = Desocupado
3 = Inactivo
4 = Menor de 10 años

AGLOMERADO N (2) Código de Aglomerado
02 = Gran La Plata
03 = Bahía Blanca - Cerri
04 = Gran Rosario
05 = Gran Santa Fé
06 = Gran Paraná
07 = Posadas
08 = Gran Resistencia
09 = Comodoro Rivadavia - Rada Tilly
10 = Gran Mendoza
12 = Corrientes
13 = Gran Córdoba
14 = Concordia
15 = Formosa
17 = Neuquén – Plottier
18 = Santiago del Estero - La Banda
19 = Jujuy - Palpalá
20 = Río Gallegos
22 = Gran Catamarca
23 = Gran Salta
25 = La Rioja
26 = Gran San Luis
27 = Gran San Juan
29 = Gran Tucumán - Tafí Viejo
30 = Santa Rosa – Toay
31 = Ushuaia - Río Grande
32 = Ciudad Autónoma de Buenos Aires
33 = Partidos del GBA
34 = Mar del Plata
36 = Río Cuarto
38 = San Nicolás – Villa Constitución
91 = Rawson – Trelew
93 = Viedma – Carmen de Patagones

In [10]:
# Mapeo del estado laboral
mapa_estado = {
    0: "No responde",
    1: "Ocupado",
    2: "Desocupado",
    3: "Inactivo",
    4: "Menor de 10 años"
}

# Mapeo de aglomerados
mapa_aglomerado = {
    2: "Gran La Plata", 3: "Bahía Blanca - Cerri", 4: "Gran Rosario",
    5: "Gran Santa Fé", 6: "Gran Paraná", 7: "Posadas", 8: "Gran Resistencia",
    9: "Comodoro Rivadavia - Rada Tilly", 10: "Gran Mendoza", 12: "Corrientes",
    13: "Gran Córdoba", 14: "Concordia", 15: "Formosa", 17: "Neuquén – Plottier",
    18: "Santiago del Estero - La Banda", 19: "Jujuy - Palpalá", 20: "Río Gallegos",
    22: "Gran Catamarca", 23: "Gran Salta", 25: "La Rioja", 26: "Gran San Luis",
    27: "Gran San Juan", 29: "Gran Tucumán - Tafí Viejo", 30: "Santa Rosa – Toay",
    31: "Ushuaia - Río Grande", 32: "CABA", 33: "Partidos del GBA",
    34: "Mar del Plata", 36: "Río Cuarto", 38: "San Nicolás – Villa Constitución",
    91: "Rawson – Trelew", 93: "Viedma – Carmen de Patagones"
}


def calcular_ocupados_desocupados(df, aglomerado=None):
    """
    Devuelve un DataFrame con cantidad de ocupados y desocupados.
    Si se especifica un aglomerado, lo filtra; si no, considera todo el país.
    """
    df = df[df['ESTADO'].isin([1, 2])].copy()

    if aglomerado is not None:
        df = df[df['AGLOMERADO'] == aglomerado]

    resumen = df.groupby('ESTADO')['PONDERA'].sum()
    
    # Crear un DataFrame con columnas explícitas para Ocupados y Desocupados
    ocupados = resumen.get(1, 0)  # .get para evitar error si falta el dato
    desocupados = resumen.get(2, 0)

    return pd.DataFrame([{
        'Ocupados': ocupados,
        'Desocupados': desocupados}])

def calcular_tasa_desempleo(df, aglomerado=None):
    """
    Calcula la tasa de desempleo como: Desocupados / (Ocupados + Desocupados).
    """
    resumen = calcular_ocupados_desocupados(df, aglomerado)
    resumen['Tasa de Desempleo (%)'] = 100 * resumen['Desocupados'] / (resumen['Ocupados'] + resumen['Desocupados'])
    return resumen['Tasa de Desempleo (%)']

def seleccionar_aglomerado(mapa_aglomerado):
    print("Seleccione un aglomerado:")
    for cod, nombre in mapa_aglomerado.items():
        print(f"{cod}: {nombre}")
    seleccion = input("Ingrese el código del aglomerado (o ENTER para todo el país): ")
    return int(seleccion) if seleccion.isdigit() else None

In [11]:
def mostrar_resumen_aglomerados(df, mapa_aglomerado):
    """
    Muestra una tabla con ocupados, desocupados y tasa de desempleo por aglomerado,
    más una fila final para el total del país.
    """
    filas = []

    for cod, nombre in mapa_aglomerado.items():
        resumen = calcular_ocupados_desocupados(df, cod)
        if not resumen.empty:
            ocupados = resumen['Ocupados'].values[0] if 'Ocupados' in resumen else 0
            desocupados = resumen['Desocupados'].values[0] if 'Desocupados' in resumen else 0
            tasa = 100 * desocupados / (ocupados + desocupados) if (ocupados + desocupados) > 0 else 0
            filas.append({
                'Aglomerado': nombre,
                'Ocupados': ocupados,
                'Desocupados': desocupados,
                'Tasa de Desempleo (%)': tasa
            })

    # Agregar total país
    resumen_total = calcular_ocupados_desocupados(df)
    ocupados_total = resumen_total['Ocupados'].values[0] if 'Ocupados' in resumen_total else 0
    desocupados_total = resumen_total['Desocupados'].values[0] if 'Desocupados' in resumen_total else 0
    tasa_total = 100 * desocupados_total / (ocupados_total + desocupados_total) if (ocupados_total + desocupados_total) > 0 else 0

    filas.append({
        'Aglomerado': 'Total País',
        'Ocupados': ocupados_total,
        'Desocupados': desocupados_total,
        'Tasa de Desempleo (%)': tasa_total
    })

    return pd.DataFrame(filas)

resumen = mostrar_resumen_aglomerados(df_filtrado, mapa_aglomerado)
resumen.sort_values(by='Aglomerado', inplace=True)

# Mostrar la tabla con formato
pd.set_option('display.float_format', '{:,.2f}'.format)
display(resumen)


,Aglomerado,Ocupados,Desocupados,Tasa de Desempleo (%)
1,Bahía Blanca - Cerri,142612,11540,7.49
25,CABA,1529661,86293,5.34
7,Comodoro Rivadavia - Rada Tilly,97440,2772,2.77
11,Concordia,64490,2854,4.24
9,Corrientes,157867,12798,7.50
12,Formosa,106388,3116,2.85
17,Gran Catamarca,102733,4150,3.88
10,Gran Córdoba,739501,60890,7.61
0,Gran La Plata,437461,39303,8.24
8,Gran Mendoza,500788,24986,4.75


1.5.3 Informar la evolución del empleo (tasa de empleo) a lo largo del tiempo. Se debe poder fi ltrar por aglomerado y en caso de no elegir ninguno se debe calcular para todo el país.
La tasa de empleo es el cociente entre personas ocupadas y la suma de personas desocupadas más ocupadas multiplicado por 100.

In [ ]:

# 1.5.2 Informar la evolución del desempleo(tasa de desempleo) a lo largo del tiempo. Se debe poder fi ltrar por aglomerado y en caso de no elegir ninguno se debe calcular para todo el país.
# La tasa de desempleo es el cociente de personas desocupadas y la suma de personas desocupadas más ocupadas multiplicado por 100.
# 1.5.3 Informar la evolución del empleo (tasa de empleo) a lo largo del tiempo. Se debe poder fi ltrar por aglomerado y en caso de no elegir ninguno se debe calcular para todo el país.
# La tasa de empleo es el cociente entre personas ocupadas y la suma de personas desocupadas más ocupadas multiplicado por 100.
# 1.5.4 Informar para cada aglomerado el total de personas ocupadas, el porcentaje con empleo estatal, el porcentaje con empleo privado y el porcentaje de otro tipo. Considerar la ocupación principal.
# 1.5.5 Se debe obtener por aglomerado el porcentaje de la tasa de empleo y desempleo. Esta información se requiere conocer para el año y trimestre más antiguo del cual se contenga información y para el año y trimestre más actual del cual se cuenta información.
# A partir de dicha información se debe visualizar un mapa que por aglomerado muestre con el color de un punto/marca si el porcentaje aumentó o disminuyó. El usuario elegirá si desea ver tasa de empleo o desempleo:
# - Al elegir la tasa de empleo se deben ver puntos verdes en los aglomerados cuya tasa de empleo aumentó con el correr del tiempo. Rojo en el caso contrario.
# - Al elegir la tasa de desempleo se deben ver puntos rojos en los aglomerados cuya tasa de empleo aumentó con el correr del tiempo. Verde en el caso contrario.